# ASPIRE — head_v2 Finetuning Demo (WeatherAUS)

Finetunes a pretrained ASPIRE checkpoint on the Australian weather dataset
using the advanced `head_v2` mode, and compares against an XGBoost baseline.

**Task:** Predict whether it will rain tomorrow (binary classification).  
**Runtime:** GPU (T4 or better recommended).  
**Expected time:** ~5 min (atom caching) + ~10 min (training).

**What head_v2 does differently from standard finetuning:**
- Freezes BERT + semantic grounding + atom processing
- Trains intra/inter set transformers + classification head
- Adds a parallel MLP branch over raw features, ensembled with the ASPIRE cosine head
- Uses focal loss, linear warmup + cosine decay LR, class-balanced support, and val-F1 checkpoint selection

> **Kaggle API key required** for dataset download.  
> Go to kaggle.com → Account → Create New Token → upload `kaggle.json` when prompted.

## 1. Install dependencies

In [ ]:
REPO_URL = "https://github.com/eddyliu5/aspire.git"

import os
if not os.path.isdir("aspire"):
    !git clone -b finetuning {REPO_URL}
%cd aspire
!pip install -e . -q
!pip install xgboost kagglehub -q

## 2. Kaggle authentication

Upload your `kaggle.json` API token when prompted, or mount Google Drive if you already have it stored there.

In [ ]:
from google.colab import files

print("Upload your kaggle.json API token:")
uploaded = files.upload()

import os
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("Kaggle credentials saved.")

## 3. Imports & config

In [ ]:
import os, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

from aspire import AspireModel

CKPT       = "eddyliu-hf/aspire"   # HuggingFace repo id (or local path)
CKPT_FILE  = "best_model.pt"
DEVICE     = "cuda"                # change to "cpu" if no GPU
SEED       = 42
TRAIN_SIZE = 5000
TEST_SIZE  = 5000
NUM_EPOCHS = 30
BATCH_SIZE = 32
NUM_SUPPORT = 5

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 4. Load the weatherAUS dataset

In [ ]:
import kagglehub

print("Downloading weatherAUS dataset from Kaggle...")
path = kagglehub.dataset_download("jsphyg/weather-dataset-rattle-package")
csv_path = os.path.join(path, "weatherAUS.csv")
df = pd.read_csv(csv_path)

target = "RainTomorrow"

df["Date"]  = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.month
df = df.drop(columns=["Date"])

df = df.drop(columns=["Sunshine", "Evaporation", "Cloud9am", "Cloud3pm"])
df = df.dropna().reset_index(drop=True)

print(f"Rows: {len(df)}  Columns: {len(df.columns)}")
print(f"\nClass distribution:")
print(df[target].value_counts())
df.head(3)

## 5. Train / test split

In [ ]:
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size=TRAIN_SIZE, test_size=TEST_SIZE,
    random_state=SEED, stratify=y,
)
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print(f"Train: {X_train.shape}   Test: {X_test.shape}")

## 6. Feature specs

Each feature needs a `name`, `description`, `dtype` (`categorical` / `continuous`),  
and either `choices` (categorical) or `value_range` (continuous).  
**The target column must be the last entry.**

In [ ]:
feature_specs = [
    {
        "name": "Location",
        "description": "Name of the Australian weather station where the observation was recorded.",
        "dtype": "categorical",
        "choices": [
            "Adelaide", "Albany", "Albury", "AliceSprings", "BadgerysCreek",
            "Ballarat", "Bendigo", "Brisbane", "Cairns", "Canberra", "Cobar",
            "CoffsHarbour", "Dartmoor", "Darwin", "GoldCoast", "Hobart",
            "Katherine", "Launceston", "Melbourne", "MelbourneAirport",
            "Mildura", "Moree", "MountGambier", "MountGinini", "Newcastle",
            "Nhil", "NorahHead", "NorfolkIsland", "Nuriootpa", "PearceRAAF",
            "Penrith", "Perth", "PerthAirport", "Portland", "Richmond",
            "Sale", "SalmonGums", "Sydney", "SydneyAirport", "Townsville",
            "Tuggeranong", "Uluru", "WaggaWagga", "Walpole", "Watsonia",
            "Williamtown", "Witchcliffe", "Wollongong", "Woomera",
        ],
    },
    {"name": "MinTemp",       "description": "Minimum temperature recorded during the day in degrees Celsius.",                      "dtype": "continuous", "value_range": (-8.5,  33.9)},
    {"name": "MaxTemp",       "description": "Maximum temperature recorded during the day in degrees Celsius.",                      "dtype": "continuous", "value_range": (-4.8,  48.1)},
    {"name": "Rainfall",      "description": "Total rainfall recorded for the day in millimeters.",                                  "dtype": "continuous", "value_range": (0.0,  371.0)},
    {
        "name": "WindGustDir",
        "description": "Direction of the strongest wind gust during the previous 24 hours.",
        "dtype": "categorical",
        "choices": ["E","ENE","ESE","N","NE","NNE","NNW","NW","S","SE","SSE","SSW","SW","W","WNW","WSW"],
    },
    {"name": "WindGustSpeed", "description": "Speed of the strongest wind gust during the previous 24 hours in km/h.",              "dtype": "continuous", "value_range": (6.0,  135.0)},
    {
        "name": "WindDir9am",
        "description": "Direction of the wind at 9am.",
        "dtype": "categorical",
        "choices": ["E","ENE","ESE","N","NE","NNE","NNW","NW","S","SE","SSE","SSW","SW","W","WNW","WSW"],
    },
    {
        "name": "WindDir3pm",
        "description": "Direction of the wind at 3pm.",
        "dtype": "categorical",
        "choices": ["E","ENE","ESE","N","NE","NNE","NNW","NW","S","SE","SSE","SSW","SW","W","WNW","WSW"],
    },
    {"name": "WindSpeed9am",  "description": "Wind speed averaged over 10 minutes prior to 9am in km/h.",                           "dtype": "continuous", "value_range": (0.0,  130.0)},
    {"name": "WindSpeed3pm",  "description": "Wind speed averaged over 10 minutes prior to 3pm in km/h.",                           "dtype": "continuous", "value_range": (0.0,   87.0)},
    {"name": "Humidity9am",   "description": "Relative humidity percentage recorded at 9am.",                                        "dtype": "continuous", "value_range": (0.0,  100.0)},
    {"name": "Humidity3pm",   "description": "Relative humidity percentage recorded at 3pm.",                                        "dtype": "continuous", "value_range": (0.0,  100.0)},
    {"name": "Pressure9am",   "description": "Atmospheric pressure at 9am reduced to mean sea level in hPa.",                       "dtype": "continuous", "value_range": (980.5, 1041.0)},
    {"name": "Pressure3pm",   "description": "Atmospheric pressure at 3pm reduced to mean sea level in hPa.",                       "dtype": "continuous", "value_range": (977.1, 1039.6)},
    {"name": "Temp9am",       "description": "Temperature recorded at 9am in degrees Celsius.",                                      "dtype": "continuous", "value_range": (-7.2,   40.2)},
    {"name": "Temp3pm",       "description": "Temperature recorded at 3pm in degrees Celsius.",                                      "dtype": "continuous", "value_range": (-5.4,   46.7)},
    {
        "name": "RainToday",
        "description": "Whether measurable rain (>=1 mm) occurred during the previous 24 hours.",
        "dtype": "categorical",
        "choices": ["No", "Yes"],
    },
    {"name": "Month",         "description": "Month of the year extracted from the observation date, representing seasonal patterns.", "dtype": "continuous", "value_range": (1, 12)},
    {
        "name": "RainTomorrow",
        "description": "Binary target: whether at least 1 mm of rain will be recorded on the following day.",
        "dtype": "categorical",
        "choices": ["No", "Yes"],
    },
]

dataset_context = (
    "This dataset contains approximately ten years of daily weather observations "
    "collected from multiple weather stations across Australia. Each record "
    "represents weather measurements for a specific location and date, including "
    "temperature, rainfall, wind speed and direction, humidity, atmospheric "
    "pressure, and other environmental indicators recorded at different times of "
    "the day. The goal is to predict the binary target variable RainTomorrow, "
    "which indicates whether it will rain on the following day."
)

assert feature_specs[-1]["name"] == target, "Target must be last in feature_specs!"
print(f"Feature specs defined: {len(feature_specs)} features (target: '{target}')")

## 7. ASPIRE — load checkpoint & finetune (head_v2)

Key `head_v2` parameters:
- `num_support=5` — class-balanced few-shot support examples per batch
- `test_fraction=0.15` — fraction of training data held out for val-F1 checkpoint selection
- `label_descriptions` — richer natural-language descriptions for the ASPIRE cosine head

In [ ]:
    "No":  "no rain tomorrow, dry conditions expected the following day",
    "Yes": "rain tomorrow, at least one millimeter of rainfall expected the following day",
}

print("=" * 60)
print("  ASPIRE v2 Finetuning")
print("=" * 60)

t0 = time.time()
model = AspireModel.from_pretrained(
    CKPT,
    filename=CKPT_FILE,
    model_type="aspire_lite",
    device=DEVICE,
    feature_specs=feature_specs,
    dataset_context=dataset_context,
)
print(f"Checkpoint loaded from: {CKPT}")

model.fit(
    X_train, y_train,
    finetune_mode="v2",
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    num_support=NUM_SUPPORT,
    test_fraction=0.15,
    random_state=SEED,
    show_progress=True,
)
fit_time = time.time() - t0
print(f"Finetune time: {fit_time:.1f}s")

## 8. Evaluate ASPIRE on the test set

In [ ]:
print("Scoring on test set...")
aspire_preds = model.predict(X_test)
aspire_acc   = accuracy_score(y_test, aspire_preds)
aspire_f1m   = f1_score(y_test, aspire_preds, average="macro",    zero_division=0)
aspire_f1w   = f1_score(y_test, aspire_preds, average="weighted", zero_division=0)

print(f"  Accuracy        : {aspire_acc:.4f}")
print(f"  F1 macro        : {aspire_f1m:.4f}")
print(f"  F1 weighted     : {aspire_f1w:.4f}")
print()
print(classification_report(y_test, aspire_preds, zero_division=0))

## 9. XGBoost baseline

In [ ]:
from xgboost import XGBClassifier

print("=" * 60)
print("  XGBoost Baseline")
print("=" * 60)

X_tr = pd.get_dummies(X_train)
X_te = pd.get_dummies(X_test)
X_tr, X_te = X_tr.align(X_te, join="left", axis=1, fill_value=0)

label_map = {v: i for i, v in enumerate(sorted(y_train.unique()))}
inv_map   = {v: k for k, v in label_map.items()}
y_tr = y_train.map(label_map)
y_te = y_test.map(label_map)

t0 = time.time()
xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective="binary:logistic", eval_metric="logloss",
    random_state=SEED,
)
xgb.fit(X_tr, y_tr, verbose=False)
xgb_time = time.time() - t0

xgb_preds = [inv_map[p] for p in xgb.predict(X_te)]
xgb_acc   = accuracy_score(y_test, xgb_preds)
xgb_f1m   = f1_score(y_test, xgb_preds, average="macro",    zero_division=0)
xgb_f1w   = f1_score(y_test, xgb_preds, average="weighted", zero_division=0)

print(f"  Accuracy        : {xgb_acc:.4f}")
print(f"  F1 macro        : {xgb_f1m:.4f}")
print(f"  F1 weighted     : {xgb_f1w:.4f}")
print(f"  Time            : {xgb_time:.1f}s")

## 10. Summary

In [ ]:
print("=" * 60)
print("  Summary")
print("=" * 60)
print(f"  {'Model':<16} {'Accuracy':>10} {'F1 macro':>10} {'F1 weighted':>12}")
print(f"  {'─'*16} {'─'*10} {'─'*10} {'─'*12}")
print(f"  {'ASPIRE v2':<16} {aspire_acc:>10.4f} {aspire_f1m:>10.4f} {aspire_f1w:>12.4f}")
print(f"  {'XGBoost':<16} {xgb_acc:>10.4f} {xgb_f1m:>10.4f} {xgb_f1w:>12.4f}")

winner = "ASPIRE v2" if aspire_f1m > xgb_f1m else "XGBoost"
delta  = abs(aspire_f1m - xgb_f1m)
print(f"\n  {winner} wins by {delta:.4f} F1 macro.")

results = {
    "aspire_v2": {"accuracy": aspire_acc, "f1_macro": aspire_f1m, "f1_weighted": aspire_f1w, "fit_time_s": fit_time},
    "xgboost":        {"accuracy": xgb_acc,   "f1_macro": xgb_f1m,   "f1_weighted": xgb_f1w,   "fit_time_s": xgb_time},
}
results